# Lab 1: Bash in Practice

## How to use this notebook
- Open in Google Colab or Jupyter with a **Python 3 kernel** and Bash available. Code cells use `%%bash` to run Bash and `%%writefile` to save scripts; these are notebook commands, not Bash syntax.
- Run each scenario's **sample data** cell once. Complete its **your script** cell, run that cell to save the script, and then run the **try it** cell, and **additional checks** cell.
- Each Bash cell starts a new shell. Files persist, but shell variables and directory changes do not. All paths below are relative to the notebook's working directory.
- Use quoted variable expansions, a `#!/bin/bash` line, and a brief comment describing each script. Use lecture tools; Python solutions are not required.
- Sample data cells overwrite their own exercise files. Scripts should leave the input files unchanged. Assume readable input files and writable output locations.
- `bc` must be installed for Scenario 3. Check with `which bc`; if unavailable, use a course environment with `bc` installed.
- Submit this notebook with all four scripts completed and their test outputs visible.

## Requirements
1. You are expected to complete **2 problems during the 107-minute lab session**.
2. **Before you leave**, ask me to review your code — I need to record your progress.
3. Before submission, rename the file. Replace `YOURNAME` with your full name (e.g., `CSI3680-Lab1-LoriXu.ipynb`)
4. Submit the **renamed and completed** `.ipynb` file to Moodle.
  - Make sure you submit your final edited `.ipynb` file — not a blank copy.

## 1. Check a client's delivery package
### Question
A design agency needs to check that every requested client file exists and contains data before sending a delivery package. Can your script identify files that are ready, empty, or missing?

### Task
Create `check_delivery.sh`.
1. Accept one or more file paths as command-line arguments. Process them in the supplied order, including names with spaces.
2. For each path, print `READY: <path>` if it is a nonempty regular file, `EMPTY: <path>` if it is an empty regular file, or `MISSING: <path>` if it does not exist. Inputs will not be directories.
3. Print a final count in the exact format below.
4. Return exit status `0` if every file is ready, or `1` if any file is empty or missing.
5. If no arguments are supplied, print `Usage: bash check_delivery.sh FILE [FILE ...]` to standard error and return `2`.

**Lecture tools:** positional arguments, `"$@"`, quoting, `for`, file tests (`-f`, `-s`), `if`, integer arithmetic, and exit status. `-f` checks whether a path is a regular file.

### Sample data (Do not change)


In [ ]:
%%bash
mkdir -p bash_practice/delivery
echo 'Approved design and assets' > "bash_practice/delivery/Client Brief.txt"
echo 'Final artwork' > bash_practice/delivery/logo.txt
> bash_practice/delivery/invoice.txt
# receipt.txt is intentionally absent; do not create it.


### Your script (Modify this part)


In [ ]:
%%writefile check_delivery.sh
#!/bin/bash
# TODO: Check arguments, inspect each file, print counts, and return the required status.


### Try it (Do not change, run to test)


In [ ]:
%%bash
bash check_delivery.sh "bash_practice/delivery/Client Brief.txt" bash_practice/delivery/invoice.txt bash_practice/delivery/receipt.txt bash_practice/delivery/logo.txt
echo "Exit status: $?"


### Expected output
```text
READY: bash_practice/delivery/Client Brief.txt
EMPTY: bash_practice/delivery/invoice.txt
MISSING: bash_practice/delivery/receipt.txt
READY: bash_practice/delivery/logo.txt
Summary: 2 ready, 1 empty, 1 missing
Exit status: 1
```


### Additional checks (Do not change, run to test)

In [ ]:
%%bash
echo "--- All ready: expect status 0 ---"
bash check_delivery.sh "bash_practice/delivery/Client Brief.txt"
echo "Exit status: $?"

echo "--- No arguments: expect empty stdout and status 2 ---"
bash check_delivery.sh > bash_practice/usage.stdout 2> bash_practice/usage.stderr
status=$?
echo "Exit status: $status"
echo "Standard output bytes: $(wc -c < bash_practice/usage.stdout)"
cat bash_practice/usage.stderr


### Expected output
```text
--- All ready: expect status 0 ---
READY: bash_practice/delivery/Client Brief.txt
Summary: 1 ready, 0 empty, 0 missing
Exit status: 0
--- No arguments: expect empty stdout and status 2 ---
Exit status: 2
Standard output bytes: 0
usage: bash check_delivery.sh FILE [FILE ...]
```


## 2. Triage an application's error log
### Question
A support technician needs a short report of application errors before deciding whether to escalate an incident. Can your script extract the relevant lines and decide whether the error count reaches an alert threshold?

### Task
Create `triage_log.sh`, called as `bash triage_log.sh LOG_FILE THRESHOLD`.
1. Assume exactly two arguments, an existing nonempty log file, and a positive integer threshold.
2. Find lines containing `error`, ignoring case. Save the original matching lines with their original line numbers to `bash_practice/errors.txt`.
3. Print the error count. Print `ALERT: investigate now` if the count is **greater than or equal to** the threshold; otherwise print `OK: below alert threshold`.
4. Print `Saved: bash_practice/errors.txt` and return status `0` after completing the report, including when an alert is raised.
5. Overwrite the report on each run. If no lines match, create an empty report and print a count of `0`.

**Lecture tools:** `grep -i`, `-n`, `-c`, command substitution, redirection, variables, and numeric tests. Remember that `grep` returns status `1` when it finds no matches; this is a normal case here.

### Sample data


In [ ]:
%%bash
mkdir -p bash_practice
cat << 'EOF' > bash_practice/application.log
09:00 INFO Service started
09:01 ERROR Database unavailable
09:02 WARN Retry scheduled
09:03 error Payment request failed
09:04 INFO Retry succeeded
09:05 Error Email delivery failed
EOF


### Your script


In [ ]:
%%writefile triage_log.sh
#!/bin/bash
# TODO: Extract numbered error lines, count matches, and compare with the threshold.


### Try it


In [ ]:
%%bash
bash triage_log.sh bash_practice/application.log 3
cat bash_practice/errors.txt


### Expected output
```text
Errors: 3
ALERT: investigate now
Saved: bash_practice/errors.txt
2:09:01 ERROR Database unavailable
4:09:03 error Payment request failed
6:09:05 Error Email delivery failed
```
The last three lines are the exact contents of `bash_practice/errors.txt`, displayed by `cat`.

### Additional checks

In [ ]:
%%bash
echo "--- Threshold 4: expect OK ---"
bash triage_log.sh bash_practice/application.log 4
echo "--- No errors: expect count 0 and an empty report ---"
echo '09:00 INFO Service healthy' > bash_practice/healthy.log
bash triage_log.sh bash_practice/healthy.log 3
echo "Exit status: $?"
echo "Report bytes: $(wc -c < bash_practice/errors.txt)"

### Expected output
```text
--- Threshold 4: expect OK ---
Errors: 3
OK: below alert threshold
Saved: bash_practice/errors.txt
--- No errors: expect count 0 and an empty report ---
Errors: 0
OK: below alert threshold
Saved: bash_practice/errors.txt
Exit status: 0
Report bytes: 0
```

## 3. Check an event budget
### Question
A student club has a fixed event budget and a list of expenses. Can your script calculate its spending and report whether the event is within budget?

### Task
Create `event_budget.sh`, called as `bash event_budget.sh EXPENSE_FILE BUDGET`.
1. Assume exactly two arguments. The expense file contains one or more valid rows in the form `item amount`; item names have no spaces and amounts are positive numbers with two decimal places. Lines beginning with `#` are comments. There are no blank lines.
2. Ignore comment lines. Count the expense rows and use `awk` to add the amounts in column 2.
3. Use `bc` to calculate `remaining = budget - total` and compare the total against the budget. The budget is a positive number with two decimal places.
4. Print the report below. A negative remaining amount means overspending. Print `OVER BUDGET` when total exceeds budget; otherwise print `WITHIN BUDGET`.
5. Return status `0` when the report is complete.

**Lecture tools:** `grep -v`, pipes, `awk` column selection and accumulation, `wc -l`, command substitution, `bc`, and `if`. Numeric values are checked by value: extra trailing zeros are acceptable; currency formatting is not required.

### Sample data


In [ ]:
%%bash
mkdir -p bash_practice
cat << 'EOF' > bash_practice/expenses.txt
# Item Amount_USD
Room 120.00
Snacks 48.50
Printing 16.25
Supplies 35.25
EOF


### Your script


In [ ]:
%%writefile event_budget.sh
#!/bin/bash
# TODO: Filter comments, count and total expenses, then calculate and report the balance.


### Try it


In [ ]:
%%bash
bash event_budget.sh bash_practice/expenses.txt 200.00


### Expected output
```text
Expenses: 4
Budget: 200.00 USD
Total: 220.00 USD
Remaining: -20.00 USD
Status: OVER BUDGET
```
`220` or `220.0` is also acceptable for `220.00`, and the same rule applies to other numeric values. Ignore leading whitespace from `wc`.


## Additional checks

In [ ]:
%%bash
echo "--- Budget 250: expect 30 remaining, WITHIN BUDGET ---"
bash event_budget.sh bash_practice/expenses.txt 250.00
echo "--- Budget 220: expect zero remaining, WITHIN BUDGET ---"
bash event_budget.sh bash_practice/expenses.txt 220.00

### Expected output
```text
--- Budget 250: expect 30 remaining, WITHIN BUDGET ---
Expenses: 4
Budget: 250.00 USD
Total: 220 USD
Remaining: 30.00 USD
WITHIN BUDGET
--- Budget 220: expect zero remaining, WITHIN BUDGET ---
Expenses: 4
Budget: 220.00 USD
Total: 220 USD
Remaining: 0 USD
WITHIN BUDGET
```

## 4. Prepare a deployment configuration
### Question
A team uses one configuration template for multiple application environments. Can your script generate a staging configuration and a short handoff note without changing the template?

### Task
Create `prepare_config.sh`, called as `bash prepare_config.sh ENV_NAME PORT`.
1. Assume exactly two valid arguments: an environment name containing only lowercase letters and digits, and a numeric port from 1024 to 65535. No argument validation is required.
2. Read `bash_practice/config.template`. Remove lines beginning with `#` and replace **every occurrence** of `__ENV__` and `__PORT__` with the arguments, using `sed`.
3. Write the result to `bash_practice/app.conf`; preserve the template.
4. Use a here document with variable expansion to create `bash_practice/handoff.txt`, with the exact text shown below for the sample invocation.
5. Print the two saved-file messages below. Overwrite both output files on each run and return status `0`.

**Lecture tools:** arguments, quoted variables, `grep -v` or `sed` deletion, global `sed` substitution, pipes, redirection, and here documents.

### Sample data


In [ ]:
%%bash
mkdir -p bash_practice
cat << 'EOF' > bash_practice/config.template
# Application deployment template
APP_NAME=club-events
ENVIRONMENT=__ENV__
PORT=__PORT__
LOG_PATH=/var/log/__ENV__/app.log
LABEL=__ENV__-__ENV__
EOF


### Your script


In [ ]:
%%writefile prepare_config.sh
#!/bin/bash
# TODO: Generate the configuration with sed and the handoff note with a here document.


### Try it


In [ ]:
%%bash
bash prepare_config.sh staging 8080
cat bash_practice/app.conf
cat bash_practice/handoff.txt


### Expected output
```text
Saved: bash_practice/app.conf
Saved: bash_practice/handoff.txt
APP_NAME=club-events
ENVIRONMENT=staging
PORT=8080
LOG_PATH=/var/log/staging/app.log
LABEL=staging-staging
Deployment handoff
Environment: staging
Port: 8080
Configuration: bash_practice/app.conf
```
**Expected files:** `app.conf` contains the five lines from `APP_NAME` through `LABEL`. `handoff.txt` contains the four lines from `Deployment handoff` through `Configuration`. The template still contains its comment and placeholders.


## Additional checks

In [ ]:
echo "--- Second deployment: expect production and 9090 ---"
bash prepare_config.sh production 9090
cat bash_practice/app.conf
cat bash_practice/handoff.txt
echo "--- Template remains unchanged ---"
cat bash_practice/config.template


### Expected output
```text
--- Second deployment: expect production and 9090 ---
Saved: bash_practice/app.conf
Saved: bash_practice/handoff.txt
APP_NAME=club-events
ENVIRONMENT=production
PORT=9090
LOG_PATH=/var/log/production/app.log
LABEL=production-production
Deployment handoff
Environment: production
Port: 9090
Configuration: bash_practice/app.conf
--- Template remains unchanged ---
# Application deployment template
APP_NAME=club-events
ENVIRONMENT=__ENV__
PORT=__PORT__
LOG_PATH=/var/log/__ENV__/app.log
LABEL=__ENV__-__ENV__
```